# 91｜兩域 fair_reference 深入分析

只讀取 90 產生的 benchmark_table.csv，不重新執行模型。
輸出 domain gap、排名、Pareto frontier、速度–準確度–coverage 圖表。


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
ROOT=Path('/content/drive/MyDrive/姿態判定')
import json,datetime,numpy as np,pandas as pd
latest=json.loads((ROOT/'04_merged_results/latest.json').read_text(encoding='utf-8')); merged=ROOT/latest['path']
assert (merged/'benchmark_table.csv').is_file(), f'找不到 90 的結果：{merged}'
df=pd.read_csv(merged/'benchmark_table.csv')
for c in ['coverage','oks_e2e','pck10_e2e','latency_mean_ms','latency_median_ms','latency_p95_ms']: df[c]=pd.to_numeric(df[c],errors='coerce')
if 'fair_ranking_eligible' not in df: df['fair_ranking_eligible']=True
df['fair_ranking_eligible']=df['fair_ranking_eligible'].astype(str).str.lower().eq('true')
if 'source_kind' not in df: df['source_kind']='fair_reference'
print('讀取：',merged); display(df.sort_values(['dataset_id','oks_e2e'],ascending=[True,False]))


In [ ]:
# 1. COCO → SyRIP domain gap
adult=df[df.dataset_id=='coco_singleperson_val2017'].set_index('model_id'); infant=df[df.dataset_id=='syrip_test100'].set_index('model_id')
gap_rows=[]
for model in sorted(set(adult.index)&set(infant.index)):
    a,i=adult.loc[model],infant.loc[model]; row={'model_id':model}
    for metric in ['oks_e2e','pck10_e2e','coverage']:
        av,iv=float(a[metric]),float(i[metric]); row[f'adult_{metric}']=av; row[f'infant_{metric}']=iv; row[f'{metric}_absolute_drop']=av-iv; row[f'{metric}_relative_drop']=(av-iv)/av if av else np.nan
    row['adult_latency_median_ms']=a['latency_median_ms']; row['infant_latency_median_ms']=i['latency_median_ms']; row['latency_change_ms']=i['latency_median_ms']-a['latency_median_ms']; gap_rows.append(row)
gap=pd.DataFrame(gap_rows).sort_values('oks_e2e_relative_drop'); display(gap)


In [ ]:
# 2. 各資料集排名與排名變化；精度排名可含 legacy，速度/效率只限 fair_reference。
parts=[]
for dataset,part in df.groupby('dataset_id'):
    x=part.copy(); x['accuracy_rank']=x.oks_e2e.rank(method='min',ascending=False).astype('Int64'); x['coverage_rank']=x.coverage.rank(method='min',ascending=False).astype('Int64'); fair=x[x.fair_ranking_eligible].copy(); fair['latency_rank']=fair.latency_median_ms.rank(method='min',ascending=True).astype('Int64'); fair['efficiency_score']=fair.oks_e2e/fair.latency_median_ms; fair['efficiency_rank']=fair.efficiency_score.rank(method='min',ascending=False).astype('Int64'); x=x.merge(fair[['model_id','latency_rank','efficiency_score','efficiency_rank']],on='model_id',how='left'); parts.append(x[['dataset_id','model_id','source_kind','fair_ranking_eligible','accuracy_rank','coverage_rank','latency_rank','efficiency_score','efficiency_rank']])
rankings=pd.concat(parts,ignore_index=True); rank_pivot=rankings.pivot(index='model_id',columns='dataset_id',values='accuracy_rank').reset_index()
if 'coco_singleperson_val2017' in rank_pivot.columns and 'syrip_test100' in rank_pivot.columns: rank_pivot['accuracy_rank_change']=rank_pivot['syrip_test100']-rank_pivot['coco_singleperson_val2017']
display(rankings.sort_values(['dataset_id','accuracy_rank'])); display(rank_pivot)


In [ ]:
# 3. Pareto frontier：只用新版 fair-reference，避免 legacy latency 混入。
def pareto(part):
    vals=part[['oks_e2e','latency_median_ms','coverage']].to_numpy(float); keep=[]
    for i in range(len(part)):
        dominated=False
        for j in range(len(part)):
            if i==j: continue
            no_worse=vals[j,0]>=vals[i,0] and vals[j,1]<=vals[i,1] and vals[j,2]>=vals[i,2]; strict=vals[j,0]>vals[i,0] or vals[j,1]<vals[i,1] or vals[j,2]>vals[i,2]
            if no_worse and strict: dominated=True; break
        keep.append(not dominated)
    out=part.copy(); out['pareto_optimal']=keep; return out
fair_df=df[df.fair_ranking_eligible].copy()
pareto_df=pd.concat([pareto(part) for _,part in fair_df.groupby('dataset_id')],ignore_index=True) if not fair_df.empty else pd.DataFrame()
display(pareto_df[pareto_df.pareto_optimal].sort_values(['dataset_id','oks_e2e'],ascending=[True,False]) if not pareto_df.empty else pd.DataFrame({'notice':['尚無新版 fair-reference 結果，不能畫公平 Pareto frontier。']}))


In [ ]:
# 4. 儲存結果與速度–準確度–coverage 圖
import matplotlib.pyplot as plt
stamp=datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ'); out=ROOT/'05_analysis_results'/f'analysis_{stamp}'; out.mkdir(parents=True,exist_ok=False)
df.to_csv(out/'benchmark_summary.csv',index=False,encoding='utf-8-sig'); gap.to_csv(out/'cross_domain_gap.csv',index=False,encoding='utf-8-sig'); rankings.to_csv(out/'model_rankings.csv',index=False,encoding='utf-8-sig'); rank_pivot.to_csv(out/'rank_change.csv',index=False,encoding='utf-8-sig'); pareto_df.to_csv(out/'pareto_analysis.csv',index=False,encoding='utf-8-sig')
for dataset,part in fair_df.groupby('dataset_id'):
    plt.figure(figsize=(8,6)); plt.scatter(part.latency_median_ms,part.oks_e2e,s=250*part.coverage.clip(lower=.05),alpha=.8)
    for _,r in part.iterrows(): plt.annotate(r.model_id,(r.latency_median_ms,r.oks_e2e),xytext=(5,5),textcoords='offset points',fontsize=8)
    plt.xlabel('Median latency (ms)'); plt.ylabel('End-to-end OKS'); plt.title(f'{dataset}: accuracy–latency–coverage'); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(out/f'{dataset}_accuracy_latency_coverage.png',dpi=180); plt.show(); plt.close()
manifest={'source':str(merged.relative_to(ROOT)),'profile_id':'mixed_new_and_legacy_backup','rerun_models':False,'outputs':['benchmark_summary.csv','cross_domain_gap.csv','model_rankings.csv','rank_change.csv','pareto_analysis.csv'],'notes':['OKS/PCK end-to-end 將失敗樣本計為 0。','準確度欄位可包含 legacy_backup，必須查看 source_kind。','Latency、efficiency 與 Pareto 僅比較 fair_ranking_eligible=True 的新版結果。']}
(out/'analysis_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8'); print('✅ 分析完成：',out)
